# 持久化机制和可恢复执行

注意避免误区：
- 这里的持久化执行和之前的重试机制、小的节点重试不一样的概念，前面的重试是小的node重试
- 这里的持久化执行是整个任务，整个任务都要有持久化的记忆机制，都要有可恢复执行的的效果

前面的是比较小的点，现在讲的是全局的持久化和可恢复  
定义：一次任务执行中的关键进度保存到可靠存储当中，  可靠存储 = 保存在磁盘当中/ 实际生产开发在数据库里，  更可靠， 作用是，不管中断，失败了，或者等待外部继续执行，  都可以拿之间的进度状态再重新恢复执行

指的是整个任务，不是单个节点， 是更大的持久化机制和可恢复执行机制

```
普通执行：
开始 -> A -> B -> C
如果执行到 B 后程序挂了，重启后可能只能从 A 重新开始。

可恢复执行：
开始 -> A   保存检查点
     -> B   保存检查点
     -> C
如果执行到 B 后挂了，恢复时可以从已保存的位置继续。
```

上面是例子


这里和之前讲的最大不同： 目的不再是保存计算过程的数据本身， 而是让整个系统在执行过程中具有可恢复性

所以检查点主要记录，任务执行到哪里了，当前状态是什么，接下来应该继续执行什么，怎么在中断的地方，继续往下执行，主要做这些操作


实际使用的时候，工作流的可恢复执行和 node的缓存机制不冲突，可以同时使用

只是前面讲的内容是单独对add_node单独设置，这里是对整个图进行设置，不一样

# 持久化机制

必须先了解持久化机制，是递进关系

langgraph持久化机制，就是在图执行过程中，每一个关键阶段的图状态保存为检查点，检查点根据线程进行组织记录，

这里的保存，不是对某个变量保存，而是保存一次图执行所需的全部状态信息，后面恢复就能继续执行了，相当于打游戏的暂停键，打boss的检查点

涉及的几个问题
1. 保存的是什么：  是当前时刻的状态快照，不用纠结当前保存了什么内容，和缓存不一样（记录进来状态和出去结果）
2. 保存在哪，开发InMemorySaver在内存（关闭后状态丢失），生产环境PostgresSaver用数据库存储，永久保存 （后续在附录A里说明部署和配置）
3. 区分不同回话：thread_id，线程id来区分不同执行的线程， 就是要使用的时候，要明确告诉线程id，理解为同一个会话，有些大模型报错的内容也可以记录下来，记录为环境上下文
4. 开发者看到的：不会直接操作checkpint， 要通过get_state, get_state_history看状态快照，不用关心底层怎么存储的，只用关注使用流程

重点：
1. 如何启用恢复执行
2. 如何配置持久化模式
3. 如何查看历史检查点
4. 如何利用检查点进行恢复、回放和分叉


## 核心组件
1. State：图状态结构
2. Channel：状态底层是Channel， 每一个数据用Channel管理
3. Checkpoint： 专门持久化存储
4. CheckpointMetada： 描述检查点内容，元数据，超步编号，父检查点id
5. checkpointer：检查点存储器，后端存在哪里，测试存在内存，生产存在数据库
6. thread：线程，但这里的概念和操作系统不一样， 这里是langgraph一条逻辑上可持久化的执行线，langgraph在操作完一次后单独记录，因为线程里的内容，会被记录在检查点里面，检查点会被记录在数据库，还能继续使用，
7. thread_id：复用线程id实现复用
8. checkpoint_ns：检查点命名空间， 额外的namespace
9. checkpoint_id： 唯一标识
10. StateSnapshot： 状态快照，用户方便观看

很多概念，后面学到什么讲什么，这里查阅

总结：
1. checkpointer 保存检查点， 内存、数据库
2. thread_id 负责唯一标识会话
3. checkpoint_id 定位某个具体历史状态
4. statesnapshot 开发者查看检查点看到的对象



# 可恢复执行
前面先讲持久化机制，再讲可恢复执行，可恢复执行依赖持久化机制

之前持久化机制把数据存储在数据库、内存，  
现在在程序退出、节点中断、人工审批暂停，恢复到之前对应的状态，利用持久化机制，只需要使用thread_id，去恢复就行


## 使用场景
1. 多轮对话：最常见场景，用户交互的时候，要调用大语言模型，如果不记录，就忘了以前，所以用户和我们沟通的时候，记录用户的线程id，我们和大预言模型沟通的时候也记录线程id
2. 中断恢复，执行到人工确认的节点，需要中断，后续恢复，再继续执行
3. 失败恢复：程序异常挂掉了，也可以基于检查点的内容，继续执行，
4. Time travel， 也可以实现回到某个历史检查点，重新播放或者分叉执行，  （分叉 fork一个新的分支， 前面的历史存储记录下来，然后分叉一个新的）


底层逻辑，根据checkpoint，检查点实现

总结：langgraph通过checkpointer持久化保存图执行过程中产生的checkpoint，并通过thread_id找回同一会话的检查点历史，从而在已有状态基础上继续执行


## 可恢复执行的具体实现

langgraph内置了持久化机制，  
1. 本地的graph，需要编译图的时候传入checkpointer，运行中的状态才会被记录下来，只用传入checkpointer，就自动生效了，
2. 然后后续想要正常工作，需要每次调用图的时候，传入threa_id，如果不传线程id，分配的是随机id，后续就不知道id是什么了，就调用不到了


存储器对象，有这些，同时实现了BaseCheckpointSaver的基类
1. InMemorySaver：内存
2. SqliteSaver：sqLite本地鸡肋数据库，可以快速实现，但不会用于生产环境
3. PostgresSaver：关系型数据库，实际生产环境用的较多（附录A介绍，要安装部署和配置）
4. MongoDBSaver：非关系型数据库
5. RedisSaver：非关系型数据库，这两款是比较老牌，java或python开发都能接触使用到的非关系型数据库，本课程不介绍  

我们使用的是PostgresSaver，后续演示


下面演示
1. 基于内存的检查点存储器
2. 持久化数据库的检查点存储器

## 基于内存的检查点存储器
保存在python进程中，进程结束则数据丢失，
- 用了jupyter只要内核没有被重启，实例没有被重新创建，检查点数据库就存在
- 如果是常规的代码执行，一旦运行到进程结束，数据就清除了

这里演示直接在内存里实现，为了演示效果，还是要和大模型交互

In [ ]:
# 先创建大预言模型，deepseek
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash", extra_body={"thinking": {"type": "disabled"}}
)


# 1. 声明状态  一般使用了大模型，就要使用MessagesState，会帮助我们不断叠加和记录数据
class OverAllState(MessagesState):
    output: str


# 2. 声明节点， 与deepseek交互
def llm_node(
    state: OverAllState,
) -> OverAllState:  # 可以优化成HumanMessage或者InputMessage，这里结构简单一点
    messages = state["messages"]

    res = model.invoke(messages)
    return {
        "messages": [
            res
        ]  # 追加的时候，使用列表，打印的时候，打印最后一条就行，不然要打印很多条
    }


def output_node(state: OverAllState) -> OverAllState:
    return {"output": state["messages"][-1]}


# 3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node", llm_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "output_node")
builder.add_edge("output_node", END)


# 上面的代码是前面学过的， 接下来给图配置checkpoint

# 4. 配置检查点存储器
checkpointer = InMemorySaver()
graph = builder.compile(
    checkpointer=checkpointer
)  # 配置了保存在内存中的检查点存储器，内容会保存


# 5. 使用的时候，要填写thread_id线程ID， 不用线程ID，有检查点也没用，是根据线程ID来确定用谁
config = {  # 可以用字典，也可以用runnableConfig
    "configurable": {"thread_id": "chapter03-01"}
}

# 6. 执行图
graph.invoke({
    "messages": [HumanMessage("你好，我是老王")]
    }, config=config)



{'messages': [HumanMessage(content='你好，我是老王', additional_kwargs={}, response_metadata={}, id='79fc481c-459e-4018-8704-dac06a78f1bd'),
  AIMessage(content='你好，老王！很高兴认识你。😊\n\n我是 DeepSeek，一个乐于助人的 AI 助手。有什么我可以帮你的吗？无论是聊聊生活、解答问题、写点东西，还是其他任何需要，尽管说！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 8, 'total_tokens': 57, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '493aed78-0055-4c63-8cc0-2ce0fad9e294', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09d91-f216-7b41-a95a-3bab65139315-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 49, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'o

In [ ]:
# 再执行，只要前面是同一个thread_id，就能记录上一次的
graph.invoke({
    "messages": [HumanMessage("你好，我是谁")]
    }, config=config) # 可以知道是老王， 因为有message记录下来了


{'messages': [HumanMessage(content='你好，我是老王', additional_kwargs={}, response_metadata={}, id='79fc481c-459e-4018-8704-dac06a78f1bd'),
  AIMessage(content='你好，老王！很高兴认识你。😊\n\n我是 DeepSeek，一个乐于助人的 AI 助手。有什么我可以帮你的吗？无论是聊聊生活、解答问题、写点东西，还是其他任何需要，尽管说！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 8, 'total_tokens': 57, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '493aed78-0055-4c63-8cc0-2ce0fad9e294', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09d91-f216-7b41-a95a-3bab65139315-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 49, 'total_tokens': 57, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  Hu

In [ ]:
# 为了演示，只有同一个id才能记录，还可以创建一个不同id的
config1 = {  # 可以用字典，也可以用runnableConfig
    "configurable": {"thread_id": "chapter03-01xx"}
}

graph.invoke({
    "messages": [HumanMessage("你好，我是谁")] # 问题一样，config不一样，已经不记得了
    }, config=config1) # 可以知道是老王， 因为有message记录下来了


{'messages': [HumanMessage(content='你好，我是谁', additional_kwargs={}, response_metadata={}, id='7f92477c-062f-4f5d-9490-6cd035203bf3'),
  AIMessage(content='你好！从我们这次对话的角度来看，我暂时还不知道你的具体身份——你对我来说是一位刚刚开始对话的朋友。\n\n如果你愿意告诉我你的名字、职业，或者今天想聊些什么，我会很乐意根据你分享的信息来更好地认识你、帮助你。😊\n\n有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 8, 'total_tokens': 69, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '9ab6d123-15bf-4634-ab7a-ba23b4d32eb3', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09d96-355f-7bb3-83ed-3cbf2a8ef8a9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 61, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}, '

总结：inmemory和内存有关，内存要是同一个，内存要是同一个，要想同一个信息， 线程id要一样，才能记录下来，否则记录不下来

这个就是内存演示的具体内容